In [9]:
# --------------------------------------------------------------------------------
# 📦 Cell 1 – Install / Imports
# --------------------------------------------------------------------------------
# If you're in a fresh environment you may need to uncomment the pip lines once.
# !pip install torchvision torch tqdm albumentations==1.4.3 opencv-python-headless

import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from matplotlib import pyplot as plt
from tqdm.auto import tqdm


In [10]:
# --------------------------------------------------------------------------------
# 📂 Cell 2 – Paths & Hyper-parameters  (tuned)
# --------------------------------------------------------------------------------
DATA_CSV   = r"C:\Users\yozev\PycharmProjects\finetuning\combined_shape16_scores.csv"
IMG_DIR    = Path(r"C:\Users\yozev\OneDrive\Desktop\Shapes2\shape16")
REF_PATH   = IMG_DIR / "original16.png"

# Training setup
BATCH_SIZE     = 32
NUM_EPOCHS     = 30
LR             = 2e-4        # ← lower
WEIGHT_DECAY   = 1e-4
VAL_FRAC       = 0.15
RANDOM_SEED    = 42
UNFREEZE_EPOCH = 5           # unfroze backbone after this epoch
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [11]:
# --------------------------------------------------------------------------------
# 🖼️ Cell 3 – Enhanced augmentation utilities
# --------------------------------------------------------------------------------
def dilate_white_line(img: np.ndarray, ksize: int = 3, iterations: int = 1):
    kernel = np.ones((ksize, ksize), np.uint8)
    return cv2.dilate(img, kernel, iterations=iterations)

class BoldLineTransform:
    def __init__(self, ksize=3, iterations=1):
        self.ksize = ksize
        self.iterations = iterations

    def __call__(self, img):
        arr = np.array(img, dtype=np.uint8)
        kernel = np.ones((self.ksize, self.ksize), np.uint8)
        dilated = cv2.dilate(arr, kernel, iterations=self.iterations)
        return Image.fromarray(dilated, mode="L")

class RotateTransform:
    def __init__(self, angle_range=(-5, 5)):
        self.angle_range = angle_range

    def __call__(self, img):
        angle = random.uniform(*self.angle_range)
        return img.rotate(angle, fillcolor=0, expand=False)

class ZoomShiftTransform:
    def __init__(self, zoom_range=(0.8, 0.95), shift_range=0.1):
        self.zoom_range = zoom_range
        self.shift_range = shift_range

    def __call__(self, img):
        w, h = img.size

        # Random zoom factor
        zoom = random.uniform(*self.zoom_range)
        new_w, new_h = int(w * zoom), int(h * zoom)

        # Resize image
        zoomed = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

        # Create new image with original size and black background
        result = Image.new('L', (w, h), 0)

        # Random shift within limits
        max_shift_x = int(w * self.shift_range)
        max_shift_y = int(h * self.shift_range)
        shift_x = random.randint(-max_shift_x, max_shift_x)
        shift_y = random.randint(-max_shift_y, max_shift_y)

        # Calculate paste position (center + shift)
        paste_x = (w - new_w) // 2 + shift_x
        paste_y = (h - new_h) // 2 + shift_y

        # Ensure paste position is within bounds
        paste_x = max(0, min(paste_x, w - new_w))
        paste_y = max(0, min(paste_y, h - new_h))

        result.paste(zoomed, (paste_x, paste_y))
        return result

# Define augmentation strategies
augmentation_transforms = {
    'bold_lines': BoldLineTransform(ksize=3, iterations=1),
    'rotate': RotateTransform(angle_range=(-5, 5)),
    'zoom_shift': ZoomShiftTransform(zoom_range=(0.8, 0.95), shift_range=0.1)
}

# Base transforms (no augmentation for validation)
child_val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5], [0.5])
])
ref_tf = child_val_tf

In [12]:
# --------------------------------------------------------------------------------
# 🗄️ Cell 4 – Balanced Augmented Dataset
# --------------------------------------------------------------------------------
class BalancedAugmentedDataset(Dataset):
    """
    Dataset that creates balanced augmented samples across all score classes.
    Generates at least min_samples_per_class for each class using augmentations.
    """
    NAME_TEMPLATE = "{id}.png"

    def __init__(self, csv_path, img_dir, ref_path, min_samples_per_class=150,
                 augmentations_per_sample=3, is_val=False):
        df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.ref_path = Path(ref_path)
        self.is_val = is_val
        self.min_samples_per_class = min_samples_per_class
        self.augmentations_per_sample = augmentations_per_sample

        assert "child_id" in df.columns, "CSV missing 'child_id'"
        assert "score" in df.columns, "CSV missing 'score'"

        # Filter out missing images
        df["img_path"] = df["child_id"].apply(
            lambda cid: self.img_dir / self.NAME_TEMPLATE.format(id=cid)
        )
        missing_mask = ~df["img_path"].apply(Path.exists)
        n_missing = missing_mask.sum()
        if n_missing:
            print(f"⚠️  Skipping {n_missing} rows with missing images.")

        self.original_data = df[~missing_mask].reset_index(drop=True)

        if not is_val:
            self.balanced_data = self._create_balanced_dataset()
            print(f"📊 Created balanced dataset with {len(self.balanced_data)} samples")
            self._print_class_distribution()
        else:
            self.balanced_data = self.original_data.copy()
            self.balanced_data['augmentation'] = 'none'

    def _create_balanced_dataset(self):
        """Create balanced dataset using augmentations."""
        # Convert scores to 0-based indexing for class analysis
        score_column = self.original_data['score'] - 1
        class_counts = score_column.value_counts().sort_index()

        print("Original class distribution:")
        for class_idx, count in class_counts.items():
            print(f"  Score {class_idx + 1}: {count} samples")

        balanced_samples = []

        for class_idx in range(7):  # scores 1-7 (0-6 in 0-based)
            class_data = self.original_data[score_column == class_idx]
            current_count = len(class_data)

            if current_count == 0:
                print(f"⚠️  No samples found for score {class_idx + 1}")
                continue

            # Add original samples
            for _, row in class_data.iterrows():
                balanced_samples.append({
                    'child_id': row['child_id'],
                    'score': row['score'],
                    'img_path': row['img_path'],
                    'augmentation': 'none'
                })

            # Calculate how many augmented samples we need
            samples_needed = max(0, self.min_samples_per_class - current_count)

            if samples_needed > 0:
                # Create augmented samples
                augmentations = ['bold_lines', 'rotate', 'zoom_shift']
                samples_per_original = samples_needed // current_count + 1

                for _, row in class_data.iterrows():
                    samples_created = 0
                    aug_cycle = 0

                    while samples_created < samples_per_original and len(balanced_samples) < (class_idx + 1) * self.min_samples_per_class:
                        aug_type = augmentations[aug_cycle % len(augmentations)]
                        balanced_samples.append({
                            'child_id': row['child_id'],
                            'score': row['score'],
                            'img_path': row['img_path'],
                            'augmentation': aug_type
                        })
                        samples_created += 1
                        aug_cycle += 1

                        if samples_created >= self.augmentations_per_sample:
                            break

        return pd.DataFrame(balanced_samples)

    def _print_class_distribution(self):
        """Print the final class distribution."""
        score_counts = (self.balanced_data['score'] - 1).value_counts().sort_index()
        print("\nFinal balanced class distribution:")
        for class_idx, count in score_counts.items():
            print(f"  Score {class_idx + 1}: {count} samples")

    def __len__(self):
        return len(self.balanced_data)

    def __getitem__(self, idx):
        row = self.balanced_data.iloc[idx]
        img_path = row["img_path"]
        score = int(row["score"]) - 1  # 1-7 → 0-6
        augmentation = row["augmentation"]

        # Load images
        child_img = Image.open(img_path).convert("L")
        ref_img = Image.open(self.ref_path).convert("L")

        # Apply augmentation if specified
        if not self.is_val and augmentation != 'none':
            if augmentation in augmentation_transforms:
                child_img = augmentation_transforms[augmentation](child_img)

        # Apply final transforms
        child_tensor = child_val_tf(child_img)
        ref_tensor = ref_tf(ref_img)

        return {
            "child": child_tensor,
            "reference": ref_tensor,
            "label": torch.tensor(score, dtype=torch.long),
        }

In [13]:
# --------------------------------------------------------------------------------
# 🧹 Cell 5 – Split dataset & build loaders with balanced augmentation
# --------------------------------------------------------------------------------
VAL_FRAC = 0.15
BATCH_SIZE = 32
RANDOM_SEED = 42
MIN_SAMPLES_PER_CLASS = 150  # Minimum samples per score class
AUGMENTATIONS_PER_SAMPLE = 3  # Max augmentations per original sample

# Create full dataset first to determine split
temp_ds = BalancedAugmentedDataset(DATA_CSV, IMG_DIR, REF_PATH,
                                  min_samples_per_class=1, is_val=True)  # Just for splitting

# Split original data indices
val_sz = int(len(temp_ds.original_data) * VAL_FRAC)
train_sz = len(temp_ds.original_data) - val_sz

train_indices, val_indices = torch.utils.data.random_split(
    range(len(temp_ds.original_data)),
    [train_sz, val_sz],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

# Create separate CSV files for train and val
train_df = temp_ds.original_data.iloc[train_indices.indices].reset_index(drop=True)
val_df = temp_ds.original_data.iloc[val_indices.indices].reset_index(drop=True)

# Save temporary CSV files
train_csv_path = "temp_train.csv"
val_csv_path = "temp_val.csv"
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)

# Create balanced training dataset and regular validation dataset
print("Creating balanced training dataset...")
train_ds = BalancedAugmentedDataset(
    train_csv_path, IMG_DIR, REF_PATH,
    min_samples_per_class=MIN_SAMPLES_PER_CLASS,
    augmentations_per_sample=AUGMENTATIONS_PER_SAMPLE,
    is_val=False
)

print("\nCreating validation dataset...")
val_ds = BalancedAugmentedDataset(
    val_csv_path, IMG_DIR, REF_PATH,
    min_samples_per_class=1,  # No augmentation for validation
    is_val=True
)

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                       shuffle=False, num_workers=0)

print(f"\n📊 Final dataset sizes:")
print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

# Clean up temporary files
os.remove(train_csv_path)
os.remove(val_csv_path)

⚠️  Skipping 2 rows with missing images.
Creating balanced training dataset...
Original class distribution:
  Score 1: 89 samples
  Score 2: 117 samples
  Score 3: 161 samples
  Score 4: 178 samples
  Score 5: 138 samples
  Score 6: 72 samples
  Score 7: 7 samples
📊 Created balanced dataset with 928 samples

Final balanced class distribution:
  Score 1: 150 samples
  Score 2: 150 samples
  Score 3: 161 samples
  Score 4: 178 samples
  Score 5: 138 samples
  Score 6: 123 samples
  Score 7: 28 samples

Creating validation dataset...

📊 Final dataset sizes:
Training samples: 928
Validation samples: 134


In [14]:
# --------------------------------------------------------------------------------
# 🧠 Cell 6 – Siamese ResNet-18 with custom tolerant loss
# --------------------------------------------------------------------------------

class ToleranceCrossEntropyLoss(nn.Module):
    """
    Custom loss that treats predictions within ±1 of the true label as correct.
    For scores outside the tolerance, uses standard cross-entropy loss.
    """
    def __init__(self, tolerance=1, reduction='mean'):
        super().__init__()
        self.tolerance = tolerance
        self.reduction = reduction
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, outputs, targets):
        # Get predicted classes
        _, predicted = outputs.max(1)

        # Calculate absolute difference between prediction and target
        diff = torch.abs(predicted - targets).float()

        # Create mask for predictions within tolerance
        within_tolerance = diff <= self.tolerance

        # Calculate standard cross-entropy loss
        ce_losses = self.ce_loss(outputs, targets)

        # Zero out losses for predictions within tolerance
        adjusted_losses = ce_losses * (~within_tolerance).float()

        if self.reduction == 'mean':
            return adjusted_losses.mean()
        elif self.reduction == 'sum':
            return adjusted_losses.sum()
        else:
            return adjusted_losses

class SiameseResNet(nn.Module):
    def __init__(self, backbone_name="resnet18", num_classes=7, freeze=True):
        super().__init__()
        backbone = models.__dict__[backbone_name](weights="IMAGENET1K_V1")
        backbone.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # 512-D
        # freeze?
        for p in self.backbone.parameters():
            p.requires_grad = not freeze
        self.classifier = nn.Sequential(
            nn.Linear(512*2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, child, reference):
        f1 = self.backbone(child).flatten(1)
        f2 = self.backbone(reference).flatten(1)
        return self.classifier(torch.cat([f1, f2], 1))

model = SiameseResNet(freeze=True).to(DEVICE)

# Use the custom tolerant loss instead of standard CrossEntropyLoss
criterion = ToleranceCrossEntropyLoss(tolerance=1)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [15]:
# --------------------------------------------------------------------------------
# 🔁 Cell 7 – Training & validation helper functions with tolerant accuracy
# --------------------------------------------------------------------------------

def calculate_tolerant_accuracy(outputs, targets, tolerance=1):
    """Calculate accuracy considering predictions within tolerance as correct."""
    _, predicted = outputs.max(1)
    within_tolerance = torch.abs(predicted - targets) <= tolerance
    return within_tolerance.float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, tolerant_correct, total = 0, 0, 0, 0
    for batch in tqdm(loader, leave=False):
        child  = batch["child"].to(DEVICE)
        ref    = batch["reference"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(child, ref)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        tolerant_correct += (torch.abs(preds - labels) <= 1).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total, tolerant_correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, tolerant_correct, total = 0, 0, 0, 0
    for batch in loader:
        child  = batch["child"].to(DEVICE)
        ref    = batch["reference"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(child, ref)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        tolerant_correct += (torch.abs(preds - labels) <= 1).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total, tolerant_correct / total

In [16]:
# --------------------------------------------------------------------------------
# 🚀 Cell 8 – Training loop with staged unfreeze & tolerant metrics
# --------------------------------------------------------------------------------
history = {"train_loss": [], "val_loss": [],
           "train_acc": [],  "val_acc": [],
           "train_tolerant_acc": [], "val_tolerant_acc": []}

best_val = 0.0
for epoch in range(1, NUM_EPOCHS+1):

    # unfreeze backbone once head has stabilised
    if epoch == UNFREEZE_EPOCH:
        for p in model.backbone.parameters():
            p.requires_grad = True
        # re-build optimizer to include backbone params
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH + 1
        )
        print(f"🔓  Unfroze backbone at epoch {epoch}")

    tr_loss, tr_acc, tr_tol_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_tol_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    history["train_tolerant_acc"].append(tr_tol_acc); history["val_tolerant_acc"].append(vl_tol_acc)

    # Use tolerant accuracy for best model selection
    is_best = vl_tol_acc > best_val
    best_val = max(best_val, vl_tol_acc)

    print(f"Epoch {epoch:02}/{NUM_EPOCHS} | "
          f"Train {tr_loss:.3f}/{tr_acc:.3f}/{tr_tol_acc:.3f} | "
          f"Val {vl_loss:.3f}/{vl_acc:.3f}/{vl_tol_acc:.3f} | "
          f"LR {scheduler.get_last_lr()[0]:.2e}"
          + ("  ✅ best so far" if is_best else ""))

  0%|          | 0/29 [00:00<?, ?it/s]

Epoch 01/30 | Train 1.098/0.166/0.441 | Val 0.620/0.284/0.642 | LR 1.99e-04  ✅ best so far


  0%|          | 0/29 [00:00<?, ?it/s]

Epoch 02/30 | Train 0.935/0.210/0.506 | Val 0.747/0.254/0.590 | LR 1.98e-04


  0%|          | 0/29 [00:00<?, ?it/s]

...

Epoch 15/30 | Train 0.191/0.548/0.844 | Val 1.480/0.209/0.485 | LR 1.24e-04


  0%|          | 0/29 [00:00<?, ?it/s]

Epoch 16/30 | Train 0.132/0.597/0.892 | Val 0.575/0.299/0.709 | LR 1.12e-04


  0%|          | 0/29 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# --------------------------------------------------------------------------------
# 📈 Cell 9 – Plot learning curves including tolerant accuracy
# --------------------------------------------------------------------------------
plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"],   label="val")
plt.ylabel("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.title("Loss")

plt.subplot(1,3,2)
plt.plot(history["train_acc"], label="train exact")
plt.plot(history["val_acc"],   label="val exact")
plt.ylabel("Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.title("Exact Accuracy")

plt.subplot(1,3,3)
plt.plot(history["train_tolerant_acc"], label="train ±1")
plt.plot(history["val_tolerant_acc"],   label="val ±1")
plt.ylabel("Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.title("Tolerant Accuracy (±1)")
plt.tight_layout()
plt.show()